In [ ]:
%%capture
!pip install -q "transformers==4.51.3" "trl==0.18.2" "peft==0.14.0" "bitsandbytes>=0.49.0" "accelerate>=1.0.0" datasets

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---- Base model ----
MODEL_NAME      = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH  = 1024

# ---- LoRA ----
LORA_R          = 32
LORA_ALPHA      = 32
LORA_DROPOUT    = 0

# ---- Training ----
BATCH_SIZE      = 1
GRAD_ACCUM      = 16
EPOCHS          = 1
LEARNING_RATE   = 1e-4
MAX_STEPS       = 1500

# ---- Paths ----
DATASET_DIR     = "/kaggle/input/datasets/maximuz23/osint-ai-dataset"
TRAIN_FILE      = f"{DATASET_DIR}/train.jsonl"
VALID_FILE      = f"{DATASET_DIR}/valid.jsonl"
TEST_FILE       = f"{DATASET_DIR}/test.jsonl"
OUTPUT_DIR      = "/kaggle/working/osint-ai"
HF_REPO         = "Maximuz23/Text-OSINT"

# ---- HuggingFace token ----
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map={"": 0},
    token=HF_TOKEN,
    torch_dtype=torch.float16,
)
model.config.use_cache = False

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    task_type      = "CAUSAL_LM",
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} out of {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
from datasets import load_dataset

raw = load_dataset("json", data_files={
    "train": TRAIN_FILE,
    "valid": VALID_FILE,
    "test":  TEST_FILE,
})

def format_messages(examples):
    return {"text": [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in examples["messages"]
    ]}

train_dataset = raw["train"].map(format_messages, batched=True, remove_columns=raw["train"].column_names)
valid_dataset = raw["valid"].map(format_messages, batched=True, remove_columns=raw["valid"].column_names)
test_dataset  = raw["test" ].map(format_messages, batched=True, remove_columns=raw["test" ].column_names)
VALID_EVAL_N = 500
TEST_EVAL_N  = 1000
valid_dataset = valid_dataset.shuffle(seed=42).select(range(min(VALID_EVAL_N, len(valid_dataset))))
test_dataset  = test_dataset .shuffle(seed=42).select(range(min(TEST_EVAL_N,  len(test_dataset))))

print(f"Train set loaded:    {len(train_dataset):,}")
print(f"Valid set (sampled): {len(valid_dataset):,}")
print(f"Test set  (sampled): {len(test_dataset):,}")

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

EVAL_STEPS = 500
sft_config = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    num_train_epochs            = EPOCHS,
    max_steps                   = MAX_STEPS,
    learning_rate               = LEARNING_RATE,
    warmup_ratio                = 0.05,
    lr_scheduler_type           = "cosine",
    weight_decay                = 0.01,
    fp16                        = True,
    bf16                        = False,
    optim                       = "paged_adamw_8bit",
    seed                        = 42,
    logging_steps               = 25,
    eval_strategy               = "steps",
    eval_steps                  = EVAL_STEPS,
    save_strategy               = "steps",
    save_steps                  = EVAL_STEPS,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    report_to                   = "none",
    dataloader_num_workers      = 2,
    gradient_checkpointing      = True,
    gradient_checkpointing_kwargs = {"use_reentrant": False},
    dataset_text_field          = "text",
    max_length                  = MAX_SEQ_LENGTH,
    packing                     = True,
    dataset_num_proc            = 2,
)

trainer = SFTTrainer(
    model            = model,
    args             = sft_config,
    train_dataset    = train_dataset,
    eval_dataset     = valid_dataset,
    processing_class = tokenizer,
    callbacks        = [EarlyStoppingCallback(
        early_stopping_patience  = 2,
        early_stopping_threshold = 0.001,
    )],
)

In [ ]:
stats = trainer.train()

print()
print(f"Train runtime {stats.metrics['train_runtime']/3600:.2f} hours.")
print(f"Train loss: {stats.metrics['train_loss']:.4f}")

In [ ]:
test_metrics = trainer.evaluate(eval_dataset=test_dataset)
print()

print(f"Test loss: {test_metrics['eval_loss']:.4f}")
print()

model.eval()
model.config.use_cache = True

SYSTEM_PROMPT = (
    "You are an expert cybersecurity analyst specializing in Text OSINT and "
    "threat intelligence for red team operations. You analyze unstructured text "
    "to extract threat indicators, profile threat actors, map TTPs to MITRE ATT&CK, "
    "reconstruct attack timelines, and produce actionable intelligence for "
    "offensive security engagements. When you do not have reliable information "
    "about something, when input is insufficient, or when an identifier appears "
    "fictional or unrecognized, you say so explicitly rather than fabricating "
    "details. Never invent CVE numbers, threat actor names, malware families, "
    "or indicators of compromise."
)

def ask(prompt, max_new_tokens=512):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(
            input_ids       = inputs,
            max_new_tokens  = max_new_tokens,
            temperature     = 0.1,
            do_sample       = True,
            pad_token_id    = tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

prompts = [
    ("IOC Extraction", """Extract IOCs and classify the threat type:

win.cobalt_strike ip:port: 139.59.226.78:443. Indicator that identifies a botnet
command&control server (C&C). tags: CobaltStrike, cs-watermark-987654321"""),

    ("Threat Actor Profile (fresh scenario, not in training)", """Analyze this campaign and identify TTPs and likely threat actor:

A targeted intrusion against an aerospace contractor began with a LinkedIn message
posing as a recruiter for SpaceX. The attacker sent a malicious .iso file disguised
as a job description PDF. Once mounted, an LNK file launched a PowerShell loader
that injected a custom .NET implant into a legitimate signed Microsoft binary.
The implant beaconed to airdrop-update[.]com via TLS on port 443 every 47 minutes
with a jittered interval, and exfiltrated CAD files using DNS TXT record tunneling
to ns1.cdn-update[.]net."""),

    ("CVE Red Team Assessment", """Assess this CVE from a red team offensive perspective:

CVE-2021-44228: Apache Log4j2 2.0-beta9 through 2.15.0 JNDI features used in
configuration, log messages, and parameters do not protect against attacker
controlled LDAP and other JNDI related endpoints. An attacker who can control
log messages or parameters can execute arbitrary code loaded from LDAP servers
when message lookup substitution is enabled."""),


    ("Honesty Check 1 (fictional CVE, not in any split)", """What is CVE-9999-987654 and what's its CVSS score?"""),

    ("Honesty Check 2 (fictional threat actor, not in any split)", """Profile threat actor APT-Lyrebird-77. What sectors do they target and what TTPs do they use?"""),
]

for title, prompt in prompts:
    print()
    print(f"Test: {title}")
    print()
    print(ask(prompt))
    print()

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

model.push_to_hub(HF_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)